# 03 · Baselines

Antes de qualquer rede neural, é preciso saber **o quanto métodos simples já resolvem**:

1. **Persistência:** a cota daqui a *h* horas = cota de agora.
2. **Regressão Ridge** (linear regularizada).
3. **Gradient boosting** (`HistGradientBoostingRegressor`).

Métrica-chave: `skill_vs_persist` (1 − MSE do modelo / MSE da persistência). **Só vale a pena se for > 0.**

**Limites contra extrapolação.** As entradas são limitadas à faixa vista no treino e a variação prevista também
(com margem `config.CLIP_MARGIN`). Modelos lineares e redes extrapolam sem freio fora dessa faixa; as árvores
já são limitadas por construção. A previsão sem limites fica salva ao lado (`pred_sem_limite_*`) para comparação no notebook 05.

In [1]:
import os, sys
sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, data, features, metrics, pipeline
plt.rcParams["figure.figsize"] = (11, 3.4)
plt.rcParams["axes.grid"] = True

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

In [2]:
panel = pipeline.load_panel()
resultados = {}

for alvo in config.TARGETS:
    if f"cota_{alvo}" not in panel:
        continue
    P = features.build_tabular(panel, alvo)
    tr, va, te = (P.masks[k] for k in ("train", "val", "test"))
    H = len(P.horizons)

    # pred_abs = previsão final (entradas e variação limitadas à faixa do treino + margem, ver config.CLIP_MARGIN)
    # livre    = mesma previsão sem os limites, guardada só para medir o efeito deles
    persist = np.repeat(P.base[:, None], H, axis=1)
    pred_abs, livre = {"persistencia": persist}, {"persistencia": persist}

    # 2) Ridge (padroniza com o treino; buracos viram a média + indicador "faltando")
    Xn, _ = features.nn_inputs(P, clip=True)
    Xn_livre, _ = features.nn_inputs(P, clip=False)
    ridge = Ridge(alpha=1.0).fit(Xn[tr], P.Y[tr])
    pred_abs["ridge"] = P.base[:, None] + features.limit_delta(P, ridge.predict(Xn))
    livre["ridge"] = P.base[:, None] + ridge.predict(Xn_livre)

    # 3) Gradient boosting: um modelo por horizonte (aceita NaN nativamente)
    Xc = features.clipped_X(P)
    delta, delta_livre = np.zeros_like(P.Y), np.zeros_like(P.Y)
    gbs = []
    for j in range(H):
        gb = HistGradientBoostingRegressor(max_depth=6, learning_rate=0.05, max_iter=300,
                                           early_stopping=False, random_state=config.RANDOM_STATE)
        gb.fit(P.X[tr], P.Y[tr][:, j])
        delta[:, j], delta_livre[:, j] = gb.predict(Xc), gb.predict(P.X)
        gbs.append(gb)
    pred_abs["hgb"] = P.base[:, None] + features.limit_delta(P, delta)
    livre["hgb"] = P.base[:, None] + delta_livre

    for nome, pa in pred_abs.items():
        pipeline.save_predictions(nome, alvo, P, pa, livre[nome])
    joblib.dump({"ridge": ridge, "hgb": gbs, "horizontes": P.horizons, "features": P.feature_names,
                 "limites_entrada": features.input_bounds(P), "limites_saida": features.output_bounds(P)},
                config.MODELS / f"baselines_{alvo}_{pipeline.current_tag()}.joblib")

    for conj, m in (("validacao", va), ("teste", te)):
        if m.sum() == 0:
            continue
        for nome, pa in pred_abs.items():
            tab = metrics.summary_table(P.Yabs[m], pa[m], P.base[m], P.horizons)
            tab.insert(0, "modelo", nome); tab.insert(0, "conjunto", conj)
            resultados.setdefault(alvo, []).append(tab.reset_index())

    print(f"\n=== {config.STATIONS[alvo]['nome']} ({alvo}) ===")
    display(pd.concat(resultados[alvo]).set_index(["conjunto", "modelo", "horizonte_h"]))

[clean_panel] dados a partir de 2018-05-01; ajustes: {'cota_86510000': '2 h < 0 cm -> NaN', 'cota_86720000': '303 h < 0 cm -> NaN'}
[pipeline] cenário: com_imerg | colunas: 12
[features] colunas removidas por pouca cobertura no treino: {'cota_86472600': '0%', 'chuva_86472600': '0%'}



=== Muçum (86510000) ===


n  RMSE_cm  MAE_cm    NSE  \
conjunto  modelo       horizonte_h                                  
validacao persistencia 6             8628   40.421  23.250  0.952   
                       8             8628   50.338  29.065  0.926   
                       12            8628   66.325  37.025  0.872   
          ridge        6             8628   29.087  14.640  0.975   
                       8             8628   35.618  17.398  0.963   
                       12            8628   46.434  23.228  0.937   
          hgb          6             8628   20.712   8.885  0.987   
                       8             8628   27.595  11.736  0.978   
                       12            8628   40.008  18.061  0.953   
teste     persistencia 6            29456   54.163  30.958  0.947   
                       8            29445   66.936  38.376  0.919   
                       12           29428   87.528  48.566  0.862   
          ridge        6            29456   43.756  21.718  0.966   
                       8            29445   55.965  27.447  0.944   
                       12           29428   84.541  44.708  0.872   
          hgb          6            29456   29.162  13.604  0.985   
                       8            29445   38.924  17.939  0.973   
                       12           29428   57.027  27.294  0.942   

                                    skill_vs_persist  
conjunto  modelo       horizonte_h                    
validacao persistencia 6                       0.000  
                       8                       0.000  
                       12                      0.000  
          ridge        6                       0.482  
                       8                       0.499  
                       12                      0.510  
          hgb          6                       0.737  
                       8                       0.699  
                       12                      0.636  
teste     persistencia 6                       0.000  
                       8                       0.000  
                       12                      0.000  
          ridge        6                       0.347  
                       8                       0.301  
                       12                      0.067  
          hgb          6                       0.710  
                       8                       0.662  
                       12                      0.576

[features] colunas removidas por pouca cobertura no treino: {'cota_86472600': '0%', 'chuva_86472600': '0%'}



=== Encantado (86720000) ===


n  RMSE_cm  MAE_cm    NSE  \
conjunto  modelo       horizonte_h                                  
validacao persistencia 6             8688   30.638  15.073  0.958   
                       8             8688   39.054  19.015  0.931   
                       12            8688   53.789  24.912  0.870   
          ridge        6             8688   16.986   8.094  0.987   
                       8             8688   23.641  11.177  0.975   
                       12            8688   35.419  16.183  0.943   
          hgb          6             8688   15.212   4.926  0.990   
                       8             8688   20.179   6.880  0.982   
                       12            8688   30.774  11.292  0.957   
teste     persistencia 6            24923   37.212  19.888  0.951   
                       8            24894   46.660  24.765  0.922   
                       12           24854   63.121  31.650  0.857   
          ridge        6            24923   22.453  16.234  0.982   
                       8            24894   31.122  23.038  0.965   
                       12           24854   50.001  37.597  0.910   
          hgb          6            24923   16.329   8.726  0.990   
                       8            24894   21.528  11.791  0.983   
                       12           24854   33.885  17.091  0.959   

                                    skill_vs_persist  
conjunto  modelo       horizonte_h                    
validacao persistencia 6                       0.000  
                       8                       0.000  
                       12                      0.000  
          ridge        6                       0.693  
                       8                       0.634  
                       12                      0.566  
          hgb          6                       0.753  
                       8                       0.733  
                       12                      0.673  
teste     persistencia 6                       0.000  
                       8                       0.000  
                       12                      0.000  
          ridge        6                       0.636  
                       8                       0.555  
                       12                      0.373  
          hgb          6                       0.807  
                       8                       0.787  
                       12                      0.712

In [3]:
for alvo, tabs in resultados.items():
    pd.concat(tabs).to_csv(config.OUTPUTS / f"baselines_{alvo}.csv", index=False)
print("tabelas salvas em", config.OUTPUTS)

tabelas salvas em /home/beckm/projetos/capstone/outputs


**Como ler:** `RMSE_cm`/`MAE_cm` em cm (menor é melhor). `NSE` fica alto até para a persistência (o nível é muito autocorrelacionado), então olhe o `skill_vs_persist`.
Se o gradient boosting ficar **perto da persistência** nos horizontes curtos, é um achado legítimo;